# PINN 求解氢原子前三个本征态 (1s, 2s, 3s) - 最终完善版

这个版本集成了所有优化：
1. **径向重要性采样 (Importance Sampling)**：大幅提升在原子核附近的学习效率。
2. **增强型参数**：128 神经元宽度，8000 轮训练，由 Adam 优化器配合学习率衰减。
3. **体积加权正交损失**：确保激发态与基态物理上的严格正交。
4. **独立可视化**：为每个轨道提供独立的、固定坐标系的大图，方便观察。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

In [ ]:
class AtomicNet(nn.Module):
    def __init__(self):
        super(AtomicNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 1)
        )
    def forward(self, r_coords):
        return self.net(r_coords)

In [ ]:
def get_sampling_points(N, device):
    # 重点：球坐标采样 + 径向平方法（让点更集中在0附近）
    r = 10.0 * torch.rand(N, 1).to(device) ** 2 
    phi = 2 * np.pi * torch.rand(N, 1).to(device)
    costheta = 2 * torch.rand(N, 1).to(device) - 1
    theta = torch.acos(costheta)
    
    x = r * torch.sin(theta) * torch.cos(phi)
    y = r * torch.sin(theta) * torch.sin(phi)
    z = r * torch.cos(theta)
    
    return x.requires_grad_(True), y.requires_grad_(True), z.requires_grad_(True), r

In [ ]:
def calc_pde_loss(model, x, y, z, E):
    psi = model(torch.cat([x, y, z], dim=1))
    
    def get_grad2(f, var):
        g = torch.autograd.grad(f, var, grad_outputs=torch.ones_like(f), create_graph=True)[0]
        return torch.autograd.grad(g, var, grad_outputs=torch.ones_like(g), create_graph=True)[0]

    laplacian = get_grad2(psi, x) + get_grad2(psi, y) + get_grad2(psi, z)
    r = torch.sqrt(x**2 + y**2 + z**2 + 1e-8)
    
    # -0.5 * r * Laplacian - psi = E * r * psi
    residual = -0.5 * r * laplacian - psi - E * r * psi
    return torch.mean(residual**2)

In [ ]:
def train_state(n, energy, prev_models=None):
    print(f"\n--- 训练顺序：第 {n} 态 (E={energy:.4f}) ---")
    model = AtomicNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=8e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4000, gamma=0.5)
    
    epochs = 8000
    for epoch in range(epochs + 1):
        optimizer.zero_grad()
        
        x_f, y_f, z_f, r_f = get_sampling_points(2048, device)
        
        # 1. PDE Loss
        loss_pde = calc_pde_loss(model, x_f, y_f, z_f, energy)
        
        # 2. Orthogonality Loss (与前序态)
        loss_orth = 0
        psi_val = model(torch.cat([x_f, y_f, z_f], dim=1))
        if prev_models:
            for prev_m in prev_models:
                prev_m.eval()
                with torch.no_grad():
                    psi_prev = prev_m(torch.cat([x_f, y_f, z_f], dim=1))
                # 引入体积元权重 (r^2 + 0.1)
                loss_orth += torch.mean(psi_val * psi_prev * (r_f**2 + 0.1))**2 * 1000.0
        
        # 3. Anchor Loss (中心约束)
        psi_0 = model(torch.zeros(1, 3).to(device))
        loss_anchor = torch.mean((psi_0 - 1.0)**2)
        
        total_loss = loss_pde + loss_orth + loss_anchor
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        if epoch % 2000 == 0:
            print(f"Epoch {epoch} | PDE: {loss_pde.item():.2e} | Orth: {loss_orth if isinstance(loss_orth, int) else loss_orth.item():.2e}")
            
    return model

In [ ]:
models = []
energies = [-0.5, -0.125, -0.0556]
for i, E in enumerate(energies):
    m = train_state(i+1, E, prev_models=models)
    models.append(m)

## 分步大图可视化

In [ ]:
titles = ["1s Orbital", "2s Orbital", "3s Orbital"]

for i in range(3):
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    grid = 50
    l = np.linspace(-10, 10, grid)
    X, Y, Z = np.meshgrid(l, l, l)
    pts = torch.FloatTensor(np.stack([X.flatten(), Y.flatten(), Z.flatten()], axis=1)).to(device)
    
    models[i].eval()
    with torch.no_grad():
        psi = models[i](pts).cpu().numpy().flatten()
    
    density = psi**2
    # 使用极低阈值以显示 2s/3s 的微弱外层
    mask = density > 0.001 * density.max()
    
    sc = ax.scatter(X.flatten()[mask], Y.flatten()[mask], Z.flatten()[mask], 
                    c=density[mask], cmap='magma', alpha=0.1, s=density[mask]*200)
    
    # 强制固定范围，防止错觉
    limit = 10
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_zlim(-limit, limit)
    
    ax.set_title(f"{titles[i]} Visualization", fontsize=16)
    plt.show()

In [ ]:
# 径向验证曲线
r_eval = torch.linspace(0, 12, 300).view(-1, 1).to(device)
coords_eval = torch.cat([r_eval, torch.zeros_like(r_eval), torch.zeros_like(r_eval)], dim=1)

plt.figure(figsize=(14, 8))
clrs = ['blue', 'red', 'green']
for i, m in enumerate(models):
    with torch.no_grad():
        vals = m(coords_eval).cpu().numpy().flatten()
    plt.plot(r_eval.cpu().numpy(), vals, label=titles[i], color=clrs[i], lw=2.5)
    
plt.axhline(0, color='black', alpha=0.3)
plt.title("Radial Profile Strategy Check (Nodes Identification)", fontsize=15)
plt.xlabel("r (Bohr)")
plt.ylabel("Psi(r)")
plt.legend()
plt.grid(True)
plt.show()